# 01 – Khảo sát dữ liệu UIT-ViSFD
Nạp dữ liệu, tính bảng, vẽ hình và xuất vào `results/`. Logic nằm trong `absa.eda`.

In [ ]:
from pathlib import Path

import pandas as pd

from absa import eda
from absa.data import load_split

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
TAB, FIG = ROOT / "results" / "tables", ROOT / "results" / "figures"
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
splits = {k: load_split(k, ROOT / "data" / "raw") for k in ("train", "dev", "test")}
{k: len(v) for k, v in splits.items()}

## 1. Khía cạnh và cảm xúc
Bảng đếm số lượt gán nhãn theo khía cạnh và cảm xúc cho từng tập; số khía cạnh mỗi bình luận.

In [ ]:
ap = {k: eda.aspect_polarity_table(v) for k, v in splits.items()}
for k, t in ap.items():
    eda.write_table(t, TAB / f"eda_aspect_polarity_{k}.csv", index_label="aspect")
apc = pd.DataFrame({k: eda.aspects_per_comment(v) for k, v in splits.items()}).fillna(0).astype(int)
eda.write_table(apc, TAB / "eda_aspects_per_comment.csv", index_label="n_aspects")
display(ap["train"], apc)

## 2. Số sao và cảm xúc

In [ ]:
svp = eda.star_vs_polarity(splits["train"])
eda.write_table(svp, TAB / "eda_star_vs_polarity.csv")
svp

## 3. Độ dài, trùng lặp, emoji, teencode
Độ dài tính theo âm tiết (tách khoảng trắng); trùng lặp so sánh sau chuẩn hóa và chữ thường.

In [ ]:
texts = {k: [str(c) for c in v["comment"]] for k, v in splits.items()}
length = pd.DataFrame({k: eda.length_stats(t) for k, t in texts.items()}).T
eda.write_table(length, TAB / "eda_length.csv", index_label="split")
dup = eda.duplicate_report(splits)
emo = eda.emoji_stats(texts["train"])
tee = eda.teencode_counts(texts["train"])
eda.write_json(dup, TAB / "eda_duplicates.json")
eda.write_json(emo, TAB / "eda_emoji.json")
eda.write_table(tee.rename("n_comments"), TAB / "eda_teencode.csv", index_label="term")
display(length)
print(dup)
print(emo)
tee.head(10)

## 4. Hình
Thứ tự khía cạnh cố định theo `ASPECTS`.

In [ ]:
eda.plot_aspect_freq(ap, FIG / "eda_aspect_freq.png")
eda.plot_aspect_polarity_stacked(ap["train"], FIG / "eda_aspect_polarity_stacked.png")
eda.plot_star_heatmap(svp, FIG / "eda_star_polarity_heatmap.png")
eda.plot_length_hist(texts["train"], FIG / "eda_length_hist.png")
sorted(p.name for p in FIG.glob("eda_*.png"))

## 5. Độ dài token con (tùy chọn, cần `transformers` và mạng)
PhoBERT: văn bản tách từ VnCoreNLP. ViSoBERT: văn bản thô sau `normalize`. Đã gồm token đặc biệt; dùng để chọn `max_length` (D9).

In [ ]:
try:
    from transformers import AutoTokenizer

    from absa.preprocess import normalize, preprocess_df

    rows = {}
    cache = ROOT / "data" / "processed"
    pho = preprocess_df(splits["train"], {"segment": "vncorenlp"}, cache, "train")
    tok = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
    rows["phobert"] = eda.subword_row(eda.subword_lengths(pho, tok))
    try:
        raw = [normalize(t) for t in texts["train"]]
        tok = AutoTokenizer.from_pretrained("uitnlp/visobert")
        rows["visobert"] = eda.subword_row(eda.subword_lengths(raw, tok))
    except Exception as e:  # keep the PhoBERT row if ViSoBERT fails to load
        print("ViSoBERT FAILED:", repr(e))
    sub = pd.DataFrame(rows).T
    eda.write_table(sub, TAB / "eda_subword_len.csv", index_label="model")
    display(sub)
except Exception as e:
    print("Bỏ qua ô token con:", repr(e))

## 6. Nhận xét
- Xem `eda_duplicates.json`: nếu `train_test_overlap > 0` thì chỉ ghi nhận, không xóa mẫu (giữ split gốc).
- `max_length` chọn theo D9 từ `eda_subword_len.csv`, ghi ở `docs/decisions.md`.